<a href="https://colab.research.google.com/github/RajTejani61/Langchain/blob/main/stable_diffusion_xl__ghibli_landscape.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q git+https://github.com/huggingface/diffusers
!wget https://raw.githubusercontent.com/huggingface/diffusers/main/examples/text_to_image/train_text_to_image_lora_sdxl.py
!pip install -U -q diffusers transformers accelerate peft bitsandbytes

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
output_folder = "/content/drive/MyDrive/Colab Notebooks/stable-diffusion-xl tunned on ghibli_landscape/ghibli_imagefolder"

In [ ]:
from datasets import load_dataset
from transformers import BlipProcessor, BlipForConditionalGeneration

In [ ]:
import torch
import os
import json

In [ ]:
if os.path.exists(output_folder) and len(os.listdir(output_folder)) > 0:
    print(f"Dataset already exists in your Drive at {output_folder}. Skipping processing!")
else:
    os.makedirs(output_folder, exist_ok=True)
    print("Loading models and data for captioning...")

    processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to("cuda")
    dataset = load_dataset("quyetISquiet/ghibli_landscape_dataset", split="train")

    metadata_lines = []
    prefix = "A peaceful landscape in the style of Studio Ghibli, "

    print("Processing images... grab a coffee, this will take a few minutes.")
    for i, example in enumerate(dataset):
        image = example["image"]

        # Generate Caption
        inputs = processor(image, text=prefix, return_tensors="pt").to("cuda")
        out = model.generate(**inputs, max_new_tokens=40)
        caption = processor.decode(out[0], skip_special_tokens=True)

        # Save Image
        image_filename = f"image_{i}.png"
        image.save(os.path.join(output_folder, image_filename))

        # Save Metadata
        metadata_lines.append(json.dumps({"file_name": image_filename, "text": caption}))

    # Save the final metadata file
    with open(os.path.join(output_folder, "metadata.jsonl"), "w") as f:
        f.write("\n".join(metadata_lines))

    print(f"Dataset securely saved to your Google Drive at: {output_folder}")

## Training

In [ ]:
# Cell 3: Crash-Proof Training Loop
!accelerate launch train_text_to_image_lora_sdxl.py \
  --pretrained_model_name_or_path="stabilityai/stable-diffusion-xl-base-1.0" \
  --pretrained_vae_model_name_or_path="madebyollin/sdxl-vae-fp16-fix" \
  --train_data_dir="/content/drive/MyDrive/Colab Notebooks/stable-diffusion-xl tunned on ghibli_landscape/ghibli_imagefolder" \
  --caption_column="text" \
  --resolution=1024 \
  --random_flip \
  --train_batch_size=1 \
  --num_train_epochs=1 \
  --checkpointing_steps=500 \
  --resume_from_checkpoint="latest" \
  --learning_rate=1e-4 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --mixed_precision="fp16" \
  --use_8bit_adam \
  --gradient_checkpointing \
  --output_dir="/content/drive/MyDrive/Colab Notebooks/stable-diffusion-xl tunned on ghibli_landscape/ghibli-sdxl-lora" \
  --seed=42

## Testing

In [ ]:
from diffusers import StableDiffusionXLPipeline

print("Loading base model...")
pipeline = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    prefix=None
).to("cuda")

Loading base model...


Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

In [ ]:
print("Loading your custom Ghibli weights...")
# Inject your trained weights directly from Google Drive
pipeline.load_lora_weights("/content/drive/MyDrive/Colab Notebooks/stable-diffusion-xl tunned on ghibli_landscape/ghibli-sdxl-lora/checkpoint-3000")

Loading your custom Ghibli weights...


No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new
No LoRA keys associated to CLIPTextModelWithProjection found with the prefix='text_encoder_2'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModelWithProjection related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new


In [ ]:
print("Generating image...")
prompt = "A peaceful village in the style of Studio Ghibli, lush greenery, summer clouds"
image = pipeline(prompt).images[0]


In [ ]:
# Save output to your Drive
save_path = "/content/drive/MyDrive/Colab Notebooks/stable-diffusion-xl tunned on ghibli_landscape/ghibli_output_1.png"
image.save(save_path)

print(f"Success! Image saved to your Google Drive as: {save_path}")

Success! Image saved to your Google Drive as: /content/drive/MyDrive/Colab Notebooks/stable-diffusion-xl tunned on ghibli_landscape/ghibli_output.png
